In [0]:
USE weather_openmeteo.gold;

-- Create Gold Dim City table
CREATE TABLE IF NOT EXISTS weather_openmeteo.gold.dim_city (
    city_SK BIGINT GENERATED ALWAYS AS IDENTITY (START WITH 1 INCREMENT BY 1) NOT NULL, -- Surrogate Key
    latitude DOUBLE,
    longitude DOUBLE,
    city_name STRING
)
USING DELTA;

---- Upsert Gold Dim city table with new and updated information
WITH cities AS (
    SELECT DISTINCT latitude, longitude, city FROM weather_openmeteo.gold.weather_daily_kpis
)
MERGE INTO weather_openmeteo.gold.dim_city AS target
USING cities AS source
ON target.latitude = source.latitude 
   AND target.longitude = source.longitude

WHEN MATCHED THEN
  UPDATE SET
    city_name = source.city

WHEN NOT MATCHED THEN
  INSERT (
    latitude, longitude, city_name
  )
  VALUES (
    source.latitude, source.longitude, source.city 
  );
  
-- Display dim city
SELECT * FROM weather_openmeteo.gold.dim_city;